In [ ]:
import re
import os
import numpy as np

In [ ]:
folder_path = '../logs/'

In [ ]:
def parse_training_log(filepath):
    runs = []
    current_fidelities = []
    current_losses = []

    with open(filepath) as f:
        for line in f:
            line = line.strip()

            if line.startswith("Run "):
                if current_fidelities:
                    runs.append({"fidelities": current_fidelities, "losses": current_losses})
                current_fidelities = []
                current_losses = []

            match = re.match(r"Epoch\s+\d+/\d+\s+\|\s+Loss:\s+([\+\-\d.]+)\s+\|\s+Fidelity:\s+([\d.]+)", line)
            if match:
                current_losses.append(float(match.group(1)))
                current_fidelities.append(float(match.group(2)))

        if current_fidelities:
            runs.append({"fidelities": current_fidelities, "losses": current_losses})

    return runs

In [ ]:
def print_stats(runs):
    final_fids = [r["fidelities"][-1] for r in runs]
    final_fids.sort()
    n = len(final_fids)

    print(f"Runs:   {n}")
    print(f"Min:    {min(final_fids):.6f}")
    print(f"Max:    {max(final_fids):.6f}")
    print(f"Mean:   {np.mean(final_fids):.6f}")
    print(f"Median: {np.median(final_fids):.6f}")
    print(f"Std:    {np.std(final_fids):.6f}")
    print(f">0.99:  {sum(1 for f in final_fids if f > 0.99)}/{n}")
    print(f">0.95:  {sum(1 for f in final_fids if f > 0.95)}/{n}")
    print(f">0.90:  {sum(1 for f in final_fids if f > 0.90)}/{n}")

In [ ]:
for filename in sorted(os.listdir(folder_path)):
    filepath = os.path.join(folder_path, filename)
    if os.path.isfile(filepath):
        print(f"\n{'='*50}")
        print(f"File: {filename}")
        print(f"{'='*50}")
        runs = parse_training_log(filepath)
        print_stats(runs)